# Classification

In diesem Jupyter Notebook werden verschiedene Lernalgorithmen für die Klassifikation untersucht, insbesondere die Klassifikation von handgeschriebenen Ziffern aus dem MNIST-Datensatz.

## Konfiguration

In [ ]:
# To support both python 2 and python 3
from __future__ import division, print_function, unicode_literals

# Common imports
import numpy as np
import os
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

# to make this notebook's output stable across runs
np.random.seed(42)

# Configuration for plotting with matplotlib
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Configuration for directories and files
# TO BE ADAPTED to context
DRIVE = "C:"
PROJECT_ROOT_DIR = DRIVE + "\\Users\\vital\\git-repos\\DHBW-CAS-W3M20026.1_Data-Science-Processes-and-Algorithms_WiSe_25-26\\laborarbeit"
PATH_IMAGES = os.path.join(PROJECT_ROOT_DIR, "images")
PATH_DATA = os.path.join(PROJECT_ROOT_DIR, "data")
PATH_MNIST_DATA = os.path.join(PATH_DATA, "mnist")
FILE_MNIST_DATA =  os.path.join(PATH_MNIST_DATA, "MNIST.data")
CHAPTER_ID = "classification"

#
# Def: Save function. 
#      For figures 
#
def save_fig(fig_id, tight_layout=True):
    pathdir = os.path.join(PATH_IMAGES, CHAPTER_ID)
    # path = os.path.join(PROJECT_ROOT_DIR, "_images", CHAPTER_ID, fig_id + ".png")
    path = os.path.join(pathdir, fig_id + ".png")
    if(not os.path.isdir(pathdir)):
      os.makedirs(pathdir)
    print("Speichere Grafik unter:\n",path)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format='png', dpi=300)
    
#
# Def: Write function. 
#      For MNIST data and labels ((X,y) fetched from openml) to file (default = "MNIST.data") in osdirectory
#
def writeMNISTDataToFile(osdirectory, data, labels, filename="MNIST.data"):
    if(not os.path.isdir(osdirectory)):
        os.makedirs(osdirectory)
    pathToFile = os.path.join(osdirectory, filename)
    f = open(pathToFile,"w") #open a file in write mode
    
    f.write("DATA:\n")
    xseq=[]
    for xs in data:
        xstr = ""
        for xss in xs:
            xstr = xstr + str(int(xss)).strip() + ","
        xstr = xstr[:len(xstr)-1]+"\n"
        xseq.append(xstr)
    # Care with big data! 
    # Better write line by line if xstr is getting too big for memory
    f.writelines(xseq)
    
    f.write("\n"+"LABELS:\n")
    ystr=""
    for ys in labels:
        ystr = ystr+ys.strip()+ ","
    ystr = ystr[:len(ystr)-1]
    f.write(ystr)

    f.close() #close the file

#
# Def: Read function.
#      For reading MNIST data from file (ospathtofile)
#
def readMNISTDataFromFile(ospathtofile, dataoffset=0, numberobjects=70000, labeloffset=2):
    f = open(ospathtofile,"r") #open a file in write mode
    XN = np.zeros(shape=(numberobjects,784))

    cont = f.readlines()
    # you may also want to remove whitespace characters like `\n` at the end of each line
    cont = [x.strip() for x in cont] 
    i=0
    for c in cont[dataoffset+1:numberobjects+1]:
        xstr = c.split(",")
        xarr = np.array([np.float64(xst) for xst in xstr])
        XN[i] = xarr
        i +=1
    ystr = cont[numberobjects+labeloffset+1]
    ystrsplit = ystr.split(",")
    yarr = np.array([int(yst) for yst in ystrsplit])
    f.close()
    return (XN,yarr)

## Load MNIST Data

In [ ]:
# from sklearn.datasets import fetch_mldata to load MNIST data from openml
import time
import sklearn.datasets as sklds
#from sklearn.datasets import fetch_openml

# print(__doc__)
remote = False
startmillis = int(round(time.time() * 1000))

# Load MNIST data from https://www.openml.org/d/554, or if avaliable locally from MNIST data file FILE_MNIST_DATA
if not os.path.isfile(FILE_MNIST_DATA):
    remote = True
    X,ystr = sklds.fetch_openml('mnist_784', version=1, return_X_y=True)
    writeMNISTDataToFile(PATH_MNIST_DATA, X, ystr)
    y = np.array([int(s) for s in ystr])
else:
    X,y = readMNISTDataFromFile(FILE_MNIST_DATA)

location = "openml" if remote else "disk" + ":"    
print("Time elapsed loading MNIST data from", location,(round(time.time() * 1000 - startmillis))/1000, "s")
print("MNIST data persisted on disk:", FILE_MNIST_DATA)

In [ ]:
os.system('cls')   # Windows
# os.system('clear') # Linux / OS X

print("Shape of data object array: ....... ", X.shape)
print("Shape of label array: ............. ", y.shape)
print("Data type of data object array: ... ", type(X))
print("Data type of label array: ......... ", type(y))
print("Data type of data object: ......... ", type(X[1]))
print("Data type of data object component: ", type(X[1][2]))
print("Data type of label: ............... ", type(y[1]))

print("Data object: ......... ", X[1])
print("Clasification ........ ", y[1])



In [ ]:
# chose a fixed data object index for representation
idxnot5 = 36132
print("Klasse:", y[idxnot5])
some_digit = X[idxnot5]
some_digit_image = some_digit.reshape(28, 28)
plt.imshow(some_digit_image, cmap = matplotlib.cm.binary,
           interpolation="nearest")
plt.axis("off")
save_fig("digit_plot_" + str(idxnot5))
plt.show()

In [ ]:
idx5 = 0
print("Klasse:", y[idx5])
some_digit5 = X[idx5]
some_digit5_image = some_digit5.reshape(28, 28)
plt.imshow(some_digit5_image, cmap = matplotlib.cm.binary,
           interpolation="nearest")
plt.axis("off")
save_fig("digit_plot_" + str(idx5))
plt.show()

In [ ]:
def plot_digit(data):
    image = data.reshape(28, 28)
    plt.imshow(image, cmap = matplotlib.cm.binary,
               interpolation="nearest")
    plt.axis("off")

In [ ]:
# EXTRA
def plot_digits(instances, images_per_row=10, **options):
    size = 28
    images_per_row = min(len(instances), images_per_row)
    images = [instance.reshape(size,size) for instance in instances]
    n_rows = (len(instances) - 1) // images_per_row + 1
    row_images = []
    n_empty = n_rows * images_per_row - len(instances)
    images.append(np.zeros((size, size * n_empty)))
    for row in range(n_rows):
        rimages = images[row * images_per_row : (row + 1) * images_per_row]
        row_images.append(np.concatenate(rimages, axis=1))
    image = np.concatenate(row_images, axis=0)
    plt.imshow(image, cmap = matplotlib.cm.binary, **options)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(12,12))
example_images = np.r_[X[:18000:600], X[13000:36600:600], X[30600:66000:590]]
plot_digits(example_images, images_per_row=13)
save_fig("more_digits_plot")
plt.show()

### Extrahiere Trainings- und Testdaten

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

### Permutiere / mische Trainingsdaten

In [ ]:
shuffle_index = np.random.permutation(60000)
X_train, y_train = X_train[shuffle_index], y_train[shuffle_index]


### Annotierte Trainingsdaten - zwei Beispiele

In [ ]:
some_digit_train1 = X_train[7]
some_digit_train2 = X_train[8]
some_digit_image1 = some_digit_train1.reshape(28, 28)
some_digit_image2 = some_digit_train2.reshape(28, 28)

plt.imshow(some_digit_image1, cmap = matplotlib.cm.binary, interpolation="nearest")
plt.axis("off")
save_fig("some_digit_plot1")
plt.show()
print("Annotation (training):  ", y_train[7], "\n")

plt.imshow(some_digit_image2, cmap = matplotlib.cm.binary, interpolation="nearest")
plt.axis("off")
save_fig("some_digit_plot2")
plt.show()
print("Annotation (training): ", y_train[8])

### Annotierte Testdaten - zwei Beispiele

In [ ]:
some_digit_test1 = X_test[7]
some_digit_test2 = X_test[8]
some_digit_image1 = some_digit_test1.reshape(28, 28)
some_digit_image2 = some_digit_test2.reshape(28, 28)

plt.imshow(some_digit_image1, cmap = matplotlib.cm.binary, interpolation="nearest")
plt.axis("off")
save_fig("some_digit_plot1")
plt.show()
print("Annotation (test):  ", y_test[7], "\n")

plt.imshow(some_digit_image2, cmap = matplotlib.cm.binary, interpolation="nearest")
plt.axis("off")
save_fig("some_digit_plot2")
plt.show()
print("Annotation (test): ", y_test[8])

# Binary classifier
* Klassifikator für den Test "5 oder Nicht-5": Zwei Klassen
* Nicht-ausgeglichene Verteilung der Klassen. Etwa 1:9 -> 9%, nicht 1/9

In [ ]:
y_train_5 = (y_train == 5)
y_test_5 = (y_test == 5)

print("Annotation (train): " , y_train[7], " is 5?", y_train_5[7])
print("Annotation (train): " , y_train[8], " is 5?", y_train_5[8])
print("Annotation (test):  " , y_test[7], " is 5?", y_test_5[7])
print("Annotation (test):  " , y_test[8], " is 5?", y_test_5[8])

In [ ]:
count = 0
# Count every dataset with 5 in training and test set
for i in range(len(y_train)):
    if y_train[i] == 5:
        count += 1
print(f"Number of 5s in training set: {count}")

count = 0
for i in range(len(y_test)):
    if y_test[i] == 5:
        count += 1
print(f"Number of 5s in test set: {count}")

### Beispiel: SGDClassifier (Stochastic Gradient Descent)

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(max_iter=300, random_state=42)
sgd_clf.fit(X_train, y_train_5)

In [ ]:
idx=65
print("Annotation (test):  " , y_test[idx], " is 5?", y_test_5[idx])
# print(y[36023])
# print(X[36023])
# for s in X:
#    if sgd_clf.predict([some_digit])[0] == False:
#        print(s)
# print(sgd_clf.predict([X_test[idx]]))
print(sgd_clf.predict([X_test[idx]]))
# print(type(sgd_clf.predict([some_digit])))

# Anwendung des Classifiers auf die MNIST-Daten

### Gütemaß Genauigkeit (Accuracy)

**Beachte:** Die Genauigkeit gibt an, wie groß der Anteil aller von einem Klassifikator korrekt bewerteten Objekte an der Gesamtheit aller Objekte ist.

#### Kreuzvalidierung mit vier Durchgängen (I)
Verwendung von Scikit-Learn-Methode cross_val_score

In [ ]:
from sklearn.model_selection import cross_val_score

print("Genauigkeit (Accuracy):", cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy"))

#### Kreuzvalidierung mit vier Durchgängen (II)
Verwendung einer eigenen Implementierung mit Hilfe der Klasse StratifiedKFold
* stratifizierte Stichproben: Aufteilung des Datensatzes in homogene Untergruppen (Strata) in Trainings- und Testdaten, ausgehend von der Label-Menge
    * in unserem Beispiel gibt die Label-Menge y_train_5 (mit zwei Klassen "5 (True)" und "Nicht-5 (False)") das Verteilungsverhältnis vor
* mehrere (vier) Folds mit jeweiligen Trainings- und Testdaten  

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skfolds = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)

for train_index, test_index in skfolds.split(X_train, y_train_5):
    clone_clf = clone(sgd_clf)
    
    print("Traininingsindizes: " ,train_index)
    print("Testindizes:        ", test_index)
    # print("Gemeinsame Elemente?", "Nein" if np.intersect1d(train_index,test_index).size==0 else "Ja")
    
    X_train_folds = X_train[train_index]
    y_train_folds = (y_train_5[train_index])
    X_test_fold = X_train[test_index]
    y_test_fold = (y_train_5[test_index])

    clone_clf.fit(X_train_folds, y_train_folds)
    y_pred = clone_clf.predict(X_test_fold)
    n_correct = sum(y_pred == y_test_fold)
    
    print("Genauigkeit:        ", n_correct / len(y_pred),"\n")


**Ergebnis:**

* Der Klassifikator bewertet stets mehr als 95% aller Objekte korrekt.
    * Sowohl mit cross_val_score als auch mit StratifiedKFold werden Genauigkeiten von über 95% erzielt!
* Gutes Ergebnis?

Nächster Schritt: Bau eines eigenen (trivialen) Klassifikators Never5 mit Ausgabe *False* für jedes Datenobjekt

### Never5Classifier

In [ ]:
from sklearn.base import BaseEstimator
class Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        pass
    def predict(self, X):
        return np.zeros((len(X), 1), dtype=bool)

### Vergleich SGDClassifier mit Never5Classifier

In [ ]:
never_5_clf = Never5Classifier()
print("Genauigkeit (SGDClassifier):   ", cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy"))
print("Genauigkeit (Never5Classifier):", cross_val_score(never_5_clf, X_train, y_train_5, cv=3, scoring="accuracy"))

* Never5Classifier bewertet mindestens 90% aller Objekte korrekt.
* Warum hat Never5Classifier eine so hohe Genauigkeit?

## Wahrheitsmatrix
* Mit der Wahrheitsmatrix können wesentlich feinere Details über die Vorhersage eines binären Klassifikators erzielt werden.

* Zuerst wird im Folgenden die Methode cross_val_predict verwendet, um ein Modell mit Hilfe von Kreuzvalidierung zu erzeugen und mit dem Modell eine Klassen-Vorhersage auf den gesamten Trainingsdaten zu ermitteln 
    * Trainiert sgd_cls mit Trainingsdatensatz in Kreuzvalidierung 
    * Erzeugt "bestes Modell" und wendet das Modell auf den gesamten Trainingsdatensatz an
    * Output: Klassen-Vorhersage auf den gesamten Trainingsdaten 
    * **Beachte:** Das ist keine Evaluation auf den Testdaten    

In [ ]:
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)

In [ ]:
from sklearn.metrics import confusion_matrix

print("Wahrheitsmatrix:")
print("-----------")
print("| tn | fp |")
print("| fn | tp |")
print("-----------")

cm = confusion_matrix(y_train_5, y_train_pred)
tn = cm[0,0]
tp = cm[1,1]
fn = cm[1,0]
fp = cm[0,1]

print(cm,"\n")
print("... oder ...\n")
print("tn =", tn)
print("tp =", tp)
print("fn =", fn)
print("fp =", fp)

### [  
#### Wahrheitsmatrix in der perfekten Welt

In [ ]:
y_train_perfect_predictions = y_train_5

In [ ]:
print("Wahrheitsmatrix (perfekte Welt): ")
perfcm = confusion_matrix(y_train_5, y_train_perfect_predictions)
print(perfcm)

### ]

## Relevanz, Sensitivität, F-Score

### Gütemaß Relevanz (Precision)

In [ ]:
from sklearn.metrics import precision_score, recall_score
print("Relevanz:", precision_score(y_train_5, y_train_pred))

In [ ]:
print ("... oder explizit ...\n")
print("Relevanz: tp /(tp + fp) =", tp/(tp+fp))

**Interpretation:** 73% der vom Klassifikator als 5 erkannten Bilder sind tatsächlich ein 5-Bild

### Gütemaß Sensitivität (Recall)

In [ ]:
print("Sensitivität:", recall_score(y_train_5, y_train_pred))

In [ ]:
print ("... oder explizit ...\n")
print("Sensitivität: tp /(tp + fn) =", tp/(tp+fn))

**Interpretation:** Von allen 5-Bildern werden 81% als solche vom Klassifikator richtigerweise erkannt

### Gütemaß F-Score

* Harmonisches Mittel von Relevanz und Sensitivität
* F-Score nimmt große Werte an, wenn Relevanz und Sensitivität groß genug sind
* Nützliches Gütemaß für unausgeglichene Klassenverteilungen 

In [ ]:
from sklearn.metrics import f1_score
print("F1-Score:", f1_score(y_train_5, y_train_pred))


In [ ]:
print ("... oder explizit ...\n")
print("F1-Score = 2tp /(2tp + fp + fn) =", 2*tp/(2*tp+fp+fn))
#4344 / (4344 + (1077 + 1307)/2)

In [ ]:
y_scores = sgd_clf.decision_function([some_digit])
print("some_digit ist Bild mit Annotation ungleich 5")
print("Score:", y_scores)
print("Klassifikation:", sgd_clf.predict([some_digit])[0])

In [ ]:
y_scores5 = sgd_clf.decision_function([some_digit5])
print("some_digit5 ist Bild mit Annotation 5")
print("Score:", y_scores5)
print("Klassifikation:", sgd_clf.predict([some_digit5])[0])

In [ ]:
threshold = 0
y_some_digit_pred = (y_scores > threshold)

In [ ]:
y_some_digit_pred

In [ ]:
threshold = 200000
y_some_digit_pred = (y_scores > threshold)
y_some_digit_pred

# Multiclass Classification

In [ ]:
sgd_mclf = SGDClassifier(max_iter=400, random_state=42)
sgd_mclf.fit(X_train, y_train)
print("Erwartete Klasse:", y_train[36132])
print("Klassifikation:", sgd_mclf.predict([X[36132]])[0])
sgd_mclf.predict([X[36132]])

In [ ]:
some_digit_scores = sgd_mclf.decision_function([some_digit])
some_digit_scores

In [ ]:
np.argmax(some_digit_scores)

In [ ]:
print("Klassen:", sgd_mclf.classes_)

In [ ]:
sgd_mclf.classes_[5]

In [ ]:
from sklearn.multiclass import OneVsOneClassifier
ovo_clf = OneVsOneClassifier(SGDClassifier(max_iter=300, random_state=42))
ovo_clf.fit(X_train, y_train)
ovo_clf.predict([some_digit])

In [ ]:
print("Anzahl Binärer Klassifikatoren:", len(ovo_clf.estimators_))

In [ ]:
cross_val_score(sgd_mclf, X_train, y_train, cv=3, scoring="accuracy")

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float64))
cross_val_score(sgd_mclf, X_train_scaled, y_train, cv=3, scoring="accuracy")

In [ ]:
y_train_pred = cross_val_predict(sgd_mclf, X_train_scaled, y_train, cv=3)
conf_mx = confusion_matrix(y_train, y_train_pred)
print(conf_mx)

In [ ]:
def plot_confusion_matrix(matrix):
    """If you prefer color and a colorbar"""
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111)
    cax = ax.matshow(matrix)
    fig.colorbar(cax)

In [ ]:
plt.matshow(conf_mx, cmap=plt.cm.gray)
save_fig("confusion_matrix_plot", tight_layout=False)
plt.show()

In [ ]:
row_sums = conf_mx.sum(axis=1, keepdims=True)
norm_conf_mx = conf_mx / row_sums

In [ ]:
np.fill_diagonal(norm_conf_mx, 0)
plt.matshow(norm_conf_mx, cmap=plt.cm.gray)
save_fig("confusion_matrix_errors_plot", tight_layout=False)
plt.show()

In [ ]:
cl_a, cl_b = 3, 5
X_aa = X_train[(y_train == cl_a) & (y_train_pred == cl_a)]
X_ab = X_train[(y_train == cl_a) & (y_train_pred == cl_b)]
X_ba = X_train[(y_train == cl_b) & (y_train_pred == cl_a)]
X_bb = X_train[(y_train == cl_b) & (y_train_pred == cl_b)]

plt.figure(figsize=(8,8))
plt.subplot(221); plot_digits(X_aa[:25], images_per_row=5)
plt.subplot(222); plot_digits(X_ab[:25], images_per_row=5)
plt.subplot(223); plot_digits(X_ba[:25], images_per_row=5)
plt.subplot(224); plot_digits(X_bb[:25], images_per_row=5)
save_fig("error_analysis_digits_plot")
plt.show()